# Optimal neighborhood size selection

This notebook selects a local neighborhood size by minimizing eigenentropy, then evaluates PCA-based normal estimates on the Stanford Bunny point cloud.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import spatial
from tqdm.auto import tqdm, trange


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from the current working directory."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    msg = "Could not find the repository root containing pyproject.toml"
    raise FileNotFoundError(msg)


REPO_ROOT = find_repo_root()

In [ ]:
def show_point_cloud(
    points: np.ndarray,
    normals: np.ndarray | None = None,
    elev: float = 0,
    azim: float = 0,
    **kwargs: object,
) -> tuple[plt.Figure, plt.Axes]:
    """Visualize a 3D point cloud, optionally colored by its normals."""
    if normals is not None:
        unit_normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
        colors: np.ndarray | str = (unit_normals + 1) / 2
        show_colorbar = True
        alpha = 1.0
    else:
        colors = "k"
        show_colorbar = False
        alpha = 0.05

    fig = plt.figure(figsize=(5, 5))
    ax = plt.axes(projection="3d")
    scatter = ax.scatter(*points.T, s=0.25, c=colors, alpha=alpha, **kwargs)
    if show_colorbar:
        colorbar = fig.colorbar(scatter, ax=ax, pad=0, shrink=0.5)
        colorbar.solids.set_alpha(1)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")
    ax.view_init(elev=elev, azim=azim, vertical_axis="y")
    return fig, ax

In [ ]:
def estimate_normals(points: np.ndarray, k: int | np.ndarray) -> np.ndarray:
    """Estimate point normals with PCA on local neighborhoods."""
    tree = spatial.KDTree(points)
    normals = np.empty_like(points)
    neighborhood_sizes = (
        np.full(points.shape[0], k, dtype=int)
        if isinstance(k, int)
        else np.asarray(k, dtype=int)
    )
    for index, point in enumerate(tqdm(points)):
        _, neighbor_indices = tree.query(point, k=neighborhood_sizes[index])
        neighborhood = points[np.atleast_1d(neighbor_indices)]
        centered = neighborhood - np.mean(neighborhood, axis=0)
        covariance = centered.T @ centered / neighborhood_sizes[index]
        _, eigenvectors = np.linalg.eigh(covariance)
        normals[index] = eigenvectors[:, 0]
    return normals

In [ ]:
def angle_error(
    estimated_normals: np.ndarray,
    reference_normals: np.ndarray,
    *,
    oriented: bool = True,
) -> np.ndarray:
    """Return angular errors between estimated and reference normals in degrees."""
    dot_products = np.einsum("ij,ij->i", estimated_normals, reference_normals)
    if not oriented:
        dot_products = np.abs(dot_products)
    return np.rad2deg(np.arccos(np.clip(dot_products, -1.0, 1.0)))

In [ ]:
def rms_angle_error(
    estimated_normals: np.ndarray,
    reference_normals: np.ndarray,
    *,
    oriented: bool = False,
) -> float:
    """Return the root-mean-square angular error in degrees."""
    errors = angle_error(
        estimated_normals,
        reference_normals,
        oriented=oriented,
    )
    return float(np.sqrt(np.mean(errors**2)))

In [ ]:
def get_optimal_neighborhood_sizes(
    points: np.ndarray,
    scale: tuple[int, int],
    step: int = 1,
    solver: str = "eigenentropy",
) -> np.ndarray:
    """Select neighborhood sizes by minimizing a local entropy measure.

    See https://doi.org/10.5194/isprsannals-II-3-181-2014.
    """
    if solver not in {"entropy", "eigenentropy"}:
        msg = f"Unsupported solver: {solver}"
        raise ValueError(msg)

    k_min, k_max = scale
    neighborhood_sizes = np.arange(k_min, k_max + 1, step, dtype=int)
    tree = spatial.KDTree(points)
    _, neighbor_indices = tree.query(points, k=k_max, workers=-1)
    optimal_sizes = np.empty(points.shape[0], dtype=int)

    for point_index in trange(points.shape[0]):
        entropy_values = np.empty(neighborhood_sizes.size)
        for scale_index, k in enumerate(neighborhood_sizes):
            neighborhood = points[neighbor_indices[point_index, :k]]
            covariance = np.cov(neighborhood, rowvar=False)
            eigenvalues = np.linalg.eigvalsh(covariance)[::-1]
            eigenvalues = np.maximum(eigenvalues, np.finfo(float).eps)
            eigenvalues /= eigenvalues.sum()

            if solver == "entropy":
                features = np.array(
                    [
                        (eigenvalues[0] - eigenvalues[1]) / eigenvalues[0],
                        (eigenvalues[1] - eigenvalues[2]) / eigenvalues[0],
                        eigenvalues[2] / eigenvalues[0],
                    ],
                )
                features = np.maximum(features, np.finfo(float).eps)
                entropy_values[scale_index] = -np.sum(features * np.log(features))
            else:
                entropy_values[scale_index] = -np.sum(eigenvalues * np.log(eigenvalues))

        optimal_sizes[point_index] = neighborhood_sizes[np.argmin(entropy_values)]
    return optimal_sizes

In [ ]:
# Load the Stanford Bunny point cloud.
bunny = REPO_ROOT / "data" / "bunny" / "bunny100k"
points = np.loadtxt(bunny.with_suffix(".xyz"))
reference_normals = np.loadtxt(bunny.with_suffix(".normals"))
fig, ax = show_point_cloud(points, reference_normals)

In [ ]:
# Estimate normals with locally optimal neighborhood sizes.
scale = (10, 100)
step = 1
optimal_sizes = get_optimal_neighborhood_sizes(
    points,
    scale,
    step,
    solver="eigenentropy",
)
estimated_normals = estimate_normals(points, optimal_sizes)
rms_error = rms_angle_error(estimated_normals, reference_normals)
rms_error